In [0]:


# Pull the GCP credentials securely from Databricks Secrets
gcp_client_email = dbutils.secrets.get(scope="candy_rush_scope", key="client_email")
gcp_project_id = dbutils.secrets.get(scope="candy_rush_scope", key="project_id")
gcp_private_key = dbutils.secrets.get(scope="candy_rush_scope", key="private_key")
gcp_private_key_id = dbutils.secrets.get(scope="candy_rush_scope", key="private_key_id")

# Set them as a configuration.
sc = spark.sparkContext
hconf = sc._jsc.hadoopConfiguration()
hconf.set("google.cloud.auth.service.account.enable", "true")
hconf.set("fs.gs.project.id", gcp_project_id)
hconf.set("fs.gs.auth.service.account.email", gcp_client_email)
hconf.set("fs.gs.auth.service.account.private.key", gcp_private_key)
hconf.set("fs.gs.auth.service.account.private.key.id", gcp_private_key_id)



BUCKET = "gs://candy-rush-514202-openfoodfacts"
OFF_PATH = BUCKET + "/products.csv.gz"  
WHR_PATH = BUCKET + "/happiness.csv"

In [0]:
import unicodedata

FIXES = {
    "turkiye": "turkey",
    "czechia": "czech-republic",
    "taiwan-province-of-china": "taiwan",
    "hong-kong-sar-of-china": "hong-kong",
    "russian-federation": "russia",
    "republic-of-korea": "south-korea",
    "viet-nam": "vietnam",
    "cote-divoire": "cote-d-ivoire",
    "dr-congo": "democratic-republic-of-the-congo",
    "congo": "republic-of-the-congo",
    "lao-pdr": "laos",
    "republic-of-moldova": "moldova",
    "state-of-palestine": "palestine",
    "swaziland": "eswatini", 
    "gambia": "the-gambia",
}

def clean_country(name):
    
    # SPlit the accented letters into letter + accent, then drop if not plain ascii
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    # remove spaces, and make lowercase
    name = name.strip().lower()
    #replace spaces and puncuation with dashes
    for ch in [" ", "'", "(", ")", ","]:
        name = name.replace(ch, "-")
        #SPlit on the dashes then drop spaces, and make - Again
    name = "-".join(part for part in name.split("-") if part)
    # if name is in FIXES swap open food spelling , otherwise dont
    return FIXES.get(name,name)
# turn text into a number
def to_float(x):
    try:
        return float(x)
    except ValueError:
        return None
    

In [0]:
import csv

off_file = sc.textFile(OFF_PATH)
# Get the first line(header) lstrop removes BOM that some files have
off_header = off_file.first().lstrip("\ufeff")
# If header has tabs, file is tab seperated-> or assume commas
DELIM = "\t" if "\t" in off_header else "," 

def parse(line):
    if DELIM == "\t":
        return line.split("\t")
    return next(csv.reader([line], delimiter=DELIM))
#Split header
cols = [c.strip() for c in parse(off_header)]
col = {name: i for i, name in enumerate(cols)}

print("Separator:", "TAB" if DELIM == "\t" else "COMMA")
print("Columns:", len(cols))
print(cols[:45])
missing = [c for c in ["code", "product_name", "categories_tags", "food_groups_tags",
                       "countries_tags", "sugars_100g", "nova_group", "additives_n"] if c not in col]
print("Missing columns:", missing)

off_rows = (off_file
            .repartition(64)
            .filter(lambda line: not line.lstrip("\ufeff").startswith(cols[0]))
            .map(parse)
            .filter(lambda r: len(r) == len(cols)))

In [0]:
# RDD1 candy by country
# Tags that count as candy
CANDY_TAGS = set([
    "en:candies", "en:confectioneries", "en:sweets", "en:chocolate-candies",
    "en:gummies", "en:lollipops", "en:chewing-gum", "en:sugary-snacks",
])

#Retrun true if product is actually candy
def is_candy(r):
    tags = r[col["categories_tags"]].split(",") + r[col["food_groups_tags"]].split(",")
    return any(t in CANDY_TAGS for t in tags)
# Keep only candy rows
candy = off_rows.filter(is_candy)
# For one product, output one (country, product) pair for EACH country it's sold in.
def explode_countries(r):
    product = (r[col["code"]],
               r[col["product_name"]].replace(",", " "),     # commas would break our CSV output
               to_float(r[col["sugars_100g"]]),
               r[col["nova_group"]],
               r[col["additives_n"]])
    
    for tag in r[col["countries_tags"]].split(","):
        #Skip other languages,so only use english tags
        if tag.startswith("en:"):
            yield (clean_country(tag[3:]), product)
# flatMa run explode_countries on every candy row and flatten all outputs into one RDD
# cache it so its result is in emory
candy_by_country = candy.flatMap(explode_countries).cache()

print("(country, product) rows:", candy_by_country.count())
candy_by_country.take(5)

In [0]:
#RDD 2 avgerage happiness by country
import csv

whr_raw = sc.textFile(WHR_PATH)
whr_header = whr_raw.first()
whr_cols = {name: i for i, name in enumerate(next(csv.reader([whr_header])))}

C_COUNTRY = whr_cols["Country name"]
C_SCORE = whr_cols["Life evaluation (3-year average)"]

happiness_by_country = (whr_raw
                        .filter(lambda line: line!=whr_header)
                        .map(lambda line: next(csv.reader([line])))
                        .filter(lambda r: r[C_COUNTRY] != "" and to_float(r[C_SCORE]) is not None)
                        .map(lambda r: (clean_country(r[C_COUNTRY]), (to_float(r[C_SCORE]),1)))
                        .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
                        .mapValues(lambda s: round(s[0]/s[1], 3))
                        .cache()
                        )

In [0]:
# check matched countries

off_countries = set(candy_by_country.keys().distinct().collect())
whr_countries = set(happiness_by_country.keys().collect())

print("Matched: ", len(off_countries & whr_countries))
print("No Match: ", sorted(whr_countries - off_countries))

candy_by_country.join(happiness_by_country).take(3)


158 of 167 countries matched, about 95%

9 to be dropped['eswatini', 'lesotho', 'malawi', 'namibia', 'north-cyprus', 'somaliland-region', 'south-sudan', 'the-gambia', 'turkmenistan']

In [0]:
#Save both rdds to bucket

OUT_CANDY = BUCKET + "/preprocessed/candy_by_country"
OUT_HAPPY = BUCKET + "/preprocessed/happiness_by_country"

Path = sc._jvm.org.apache.hadoop.fs.Path
for out in [OUT_CANDY, OUT_HAPPY]:
    Path(out).getFileSystem(hconf).delete(Path(out), True)

candy_by_country \
    .map(lambda kv: ",".join(str(x) for x in (kv[0],) + kv[1])) \
    .saveAsTextFile(OUT_CANDY)

happiness_by_country \
    .map(lambda kv: f"{kv[0]}, {kv[1]}") \
    .coalesce(1) \
    .saveAsTextFile(OUT_HAPPY)

# read a few lines back to show the save worked
print(sc.textFile(OUT_CANDY).take(3))
print(sc.textFile(OUT_HAPPY).take(3))
